<a href="https://colab.research.google.com/github/Thiago123408/benchmark-quantizacao-cpu/blob/main/benchmark_quantizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy
import time
import torch
import torchvision.models as models

# ==========================================
# PARTE 1: CARREGAR O MODELO E OS DADOS
# ==========================================
print("📦 Carregando modelo ResNet-18 original...")
# Carrega o modelo padrão com pesos já treinados
model_fp32 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_fp32.eval()

# Cria uma imagem falsa (Tensor) no padrão esperado (Batch=1, Cores RGB=3, 224x224 pixels)
dummy_input = torch.randn(1, 3, 224, 224)
print("✅ Modelo e dados prontos para o teste!")


# ==========================================
# PARTE 2: APLICAR QUANTIZAÇÃO NA CONV1
# ==========================================
print("\n🔧 Aplicando quantização dinâmica especificamente na camada conv1...")
model_quant = copy.deepcopy(model_fp32)

# Mapeia e converte apenas o módulo conv1 para pesos int8 (reduzindo uso de memória)
model_quant.conv1 = torch.quantization.quantize_dynamic(
    model_fp32.conv1, {torch.nn.Conv2d}, dtype=torch.qint8
)
print("✅ Camada conv1 modificada com sucesso!")


# ==========================================
# PARTE 3: BENCHMARK (TESTE DE VELOCIDADE)
# ==========================================
# Função para medir o tempo médio de resposta (Latência)
def benchmark(model, nome_do_teste):
    # Aquecimento (Warm-up): descarta as primeiras rodadas para o sistema estabilizar
    for _ in range(10):
        _ = model(dummy_input)

    start_time = time.time()
    iteracoes = 100
    for _ in range(iteracoes):
        _ = model(dummy_input)
    end_time = time.time()

    # Calcula a média por inferência e transforma em milissegundos
    avg_latency = ((end_time - start_time) / iteracoes) * 1000
    print(f"⏱️ [{nome_do_teste}] Latência Média: {avg_latency:.2f} ms")
    return avg_latency

# Executando a comparação direta
print("\n🚀 Iniciando os testes de velocidade...\n")
latency_fp32 = benchmark(model_fp32, "Modelo Original FP32")
latency_quant = benchmark(model_quant, "Modelo Modificado (conv1 INT8)")

# Mostra a diferença percentual e a conclusão prática
mudanca = (latency_fp32 - latency_quant) / latency_fp32 * 100
if mudanca > 0:
    print(f"\n⚡ Resultado: O modelo modificado ficou {mudanca:.1f}% MAIS RÁPIDO.")
else:
    print(f"\n⚠️ Resultado: O modelo modificado ficou {abs(mudanca):.1f}% MAIS LENTO.")
    print("💡 Isso comprova o relatório: a conversão dinâmica em apenas uma camada gera overhead na CPU!")

📦 Carregando modelo ResNet-18 original...
✅ Modelo e dados prontos para o teste!

🔧 Aplicando quantização dinâmica especificamente na camada conv1...
✅ Camada conv1 modificada com sucesso!

🚀 Iniciando os testes de velocidade...



/tmp/ipykernel_653/586637596.py:26: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quant.conv1 = torch.quantization.quantize_dynamic(


⏱️ [Modelo Original FP32] Latência Média: 110.68 ms
⏱️ [Modelo Modificado (conv1 INT8)] Latência Média: 109.40 ms

⚡ Resultado: O modelo modificado ficou 1.2% MAIS RÁPIDO.
